In [1]:
import pandas as pd
import reComBat as rc
import numpy as np

In [7]:
counts = pd.read_csv(
    'psoriasis/psoriasis.all.exp.tsv',
    sep = '\t',
    index_col = 0
)
meta = pd.read_csv(
    'psoriasis/psoriasis.all.meta.tsv',
    sep = '\t',
    index_col = 0
)

In [3]:
def logcpm(data):
    return np.log1p(data.divide(data.sum(axis = 1), axis = 0) * 1e6)

norm = logcpm(counts)

In [4]:
def correct_batch(norm_counts, batches, wanted_variation):
    standardized_norm_counts = ((norm_counts.T - norm_counts.T.mean()) / norm_counts.T.std()).T

    print('correcting batches')
    model = rc.reComBat(
        parametric=True,
        model='ridge',
        config={'alpha':1e-9},
        conv_criterion=1e-4,
        max_iter=1000,
        n_jobs=4,
        mean_only=False,         
        optimize_params=True,
        reference_batch=None,
        verbose=True                
    )

    X = model.fit_transform(
        standardized_norm_counts,
        batches,
        X=wanted_variation,
    )
    return X

In [13]:
a = pd.get_dummies(meta.sra_study_acc).values
b = pd.get_dummies(meta.Disease).values
np.hstack([a, b]).shape

(321, 7)

In [5]:
corrected = correct_batch(
    norm,
    meta.sra_study_acc,
    meta.loc[:, ['Disease']]
)

[reComBat] 2026-04-17 11:57:46,603 Starting to fot reComBat.
[reComBat] 2026-04-17 11:57:46,613 Fit the linear model.


correcting batches


[reComBat] 2026-04-17 11:57:46,854 Starting the empirical parametric optimisation.
[reComBat] 2026-04-17 11:57:47,033 Optimisation finished.
[reComBat] 2026-04-17 11:57:47,035 reComBat is fitted.
[reComBat] 2026-04-17 11:57:47,036 Starting to transform.
[reComBat] 2026-04-17 11:57:47,238 Transform finished.


In [8]:
corrected.to_csv('psoriasis/psoriasis.all.exp.recombat.tsv', sep = '\t')